In [ ]:
from google.colab import files

uploaded = files.upload()
print(uploaded)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import math

# Data Cleaning

In [ ]:
ds=pd.read_csv("CSISA_IND_LDS_Rice_2018_Data.csv")
# from google.colab import files
# uploaded = files.upload()
# ds=pd.read_csv("CSISA_IND_LDS_Rice_2018_Data.csv")
ds

In [ ]:
selected_columns = ['state', 'crop','season','LLU','acreConv','cultLand','cropCultLand',
                    'cropLarestAreaAcre','soilTexture','drainClass','soilPerception','prevCrop',
                    'prevCropTillage','cropResiduePcnt','residueBurnt','varType','varName',
                    'LandPrep','CropEst','cropSeedAmt','statSHC','FYM','typeFYM','amtFYM',
                    'applyMineralFert','basalFerts','basalDAP','basalNPK','basalUrea','basalNPKS',
                    'basalMoP','basalSSP','basalTSP','basalZnSO4','basalGypsum','basalBoron',
                    'otherBasalFert','otherBasalFertAmt','firstTopDressFert','1tdDAP','1appDaysDAP',
                    '1tdNPK','1appDaysNPK','1tdUrea','1appDaysUrea','1tdNPKS','1appDaysNPKS',
                    '1tdMoP','1appDaysMoP','1tdSSP','1appDaysSSP','1tdTSP','1appDaysTSP',
                    '1tdZnSO4','1appDaysZnSO4','1tdGypsum','1appDaysGypsum','1tdBoron','1appDaysBoron',
                    '1tdOtherFert','1tdOtherFertAmt','1appDaysOtherFert','secondTopDressFert','2tdDAP','2appDaysDAP',
                    '2tdNPK','2appDaysNPK','2tdUrea','2appDaysUrea','2tdNPKS','2appDaysNPKS','2tdMoP',
                    '2appDaysMoP','2tdSSP','2appDaysSSP','2tdTSP','2appDaysTSP','2tdZnSO4','2appDaysZnSO4',
                    '2tdGypsum','2appDaysGypsum','2tdBoron','2appDaysBoron','2tdOtherFert','2tdOtherFertAmt',
                    '2appDaysOtherFert','thirdTopDressFert','3tdDAP','3appDaysDAP','3tdNPK','3appDaysNPK',
                    '3tdUrea','3appDaysUrea','3tdNPKS','3appDaysNPKS','3tdMoP','3appDaysMoP','3tdSSP',
                    '3appDaysSSP','3tdTSP','3appDaysTSP','3tdZnSO4','3appDaysZnSO4','3tdGypsum',
                    '3appDaysGypsum','3tdBoron','3appDaysBoron','3tdOtherFert','3tdOtherFertAmt',
                    '3appDaysOtherFert','gradeNPK','gradeNPKS','fertOnTime','avgDelayWeeks','irrigAvail',
                    'irrigSource','irrigGrthStage','irrigTimes','tubewellDepthFeet','pumpUse','layFlatPipe',
                    'droughtSeverity','floodSeverity','weedSeverity','insectSeverity','insecticides',
                    'insecticidesName','diseaseSeverity','pesticides','pesticidesName','lodgingPercent','herbAppTimes',
                    '1herbName','1herbAppDays','2herbName','2herbAppDays','3herbName','3herbAppDays',
                    'manualWeedTimes','1manualWeeding','2manualWeeding','3manualWeeding','allWeeds','allOtherWeeds',
                    'topTenWeeds','top1stWeed','top2ndWeed','top3rdWeed','top4thWeed','top5thWeed',
                    'harvestMethod','threshing','totalGrainYieldQUINTAL','largestPlotYieldQUNITAL','quintalPerAcre',
                    'O-largestPlotGPS-Latitude','O-largestPlotGPS-Longitude','O-largestPlotGPS-Altitude',
                    'O-largestPlotGPS-Accuracy']


In [ ]:
data = ds[selected_columns]
data

In [ ]:
data=data.drop_duplicates()
duplicate_columns = data.columns[data.columns.duplicated()]
data=data.drop(duplicate_columns, axis=1)
data

In [ ]:
columns_with_no_values = data.columns[data.isna().all()].tolist()
# Drop columns with no values
data.drop(columns=columns_with_no_values, inplace=True)
data

In [ ]:
# Fill missing values in numeric columns with 0 and ensure integer data type
numeric_columns = data.select_dtypes(include=['number']).columns
data[numeric_columns] = data[numeric_columns].fillna(0)
data[numeric_columns] = abs(data[numeric_columns])
data

In [ ]:
non_numeric_columns = data.select_dtypes(exclude=['number']).columns
# Fill missing values in non-numeric columns with None
for col in non_numeric_columns:
    data[col] = data[col].fillna('no')
data

In [ ]:
data.info(verbose=1)

In [ ]:
data.to_csv('sample.csv',index=False)

# Machine Learning

In [ ]:
data=pd.read_csv('sample.csv')
# from google.colab import files
# uploaded = files.upload()
# data=pd.read_csv("sample.csv")

In [ ]:
from sklearn.preprocessing import LabelEncoder
data=data.drop(['crop','totalGrainYieldQUINTAL','largestPlotYieldQUNITAL'],axis=1)
non_numeric_columns = data.select_dtypes(include=[object]).columns.tolist()
label_encoder = LabelEncoder()
for col in non_numeric_columns:
    data[col]=label_encoder.fit_transform(data[col])
data.info(verbose=1)

In [ ]:
shuffled_df = data.sample(frac=1, random_state=42)
sampled_data = shuffled_df.head(1000)

In [ ]:
column_index = shuffled_df.columns.get_loc('quintalPerAcre')
X=sampled_data.iloc[:,list(range(0,149))+list(range(150,154))]
y=sampled_data['quintalPerAcre']
print(X.shape,y.shape)
X = X.reset_index(drop=True)
y= y.reset_index(drop=True)
#from sklearn.model_selection import train_test_split
#xtrain, xtest, ytrain, ytest = train_test_split(X, y , test_size=0.20,random_state=42

# Dynamic Dropout Regularization

## Adam Optimizer

In [ ]:
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Layer, Conv1D, MaxPooling1D, LSTM, Flatten, Dense, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.regularizers import l2
from tensorflow.keras.callbacks import Callback
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.model_selection import KFold

kf = KFold(n_splits=5, shuffle=True,random_state=42)

import numpy as np
# from tensorflow.keras.layers import Layer
import tensorflow as tf

class DynamicDropout(Layer):
    def __init__(self, initial_rate=0.1, **kwargs):
        super(DynamicDropout, self).__init__(**kwargs)
        self.rate = tf.Variable(initial_rate, trainable=False, dtype=tf.float32)

    def call(self, inputs, training=None):
        if training:
            return tf.nn.dropout(inputs, rate=self.rate)
        return inputs

    def get_config(self):
        config = super().get_config().copy()
        config.update({"initial_rate": self.rate.numpy()})
        return config


class DynamicL2Regularization(Callback):
    def __init__(self, conv_layers, dropout_layers, initial_dropout=0.3, initial_strength=0.01):
        super(DynamicL2Regularization, self).__init__()
        self.conv_layers = conv_layers
        self.dropout_layers = dropout_layers
        # self.initial_l2 = initial_l2
        self.initial_dropout = initial_dropout
        self.initial_strength = initial_strength

    def on_epoch_end(self, epoch, logs=None):
        training_loss = logs['loss']
        d1=-training_loss
        d2=(-self.initial_strength*math.exp(-training_loss/20))/20
        gradient=d1*d2

        # Dynamic L2 and Dropout strength
        dynamic_l2_regularization = self.initial_strength - (0.001*gradient)
        dynamic_dropout=self.initial_dropout+(0.001*gradient)
        # dynamic_dropout=self.initial_dropout*math.exp(training_loss/20)

        # # Bound the values
        # dynamic_l2_regularization=max(0.001, min(0.1, dynamic_l2_regularization))
        dynamic_dropout=max(0.1, min(0.5,dynamic_dropout))


        regularization_strength_values.append(dynamic_l2_regularization)
        dropout_rate_values.append(dynamic_dropout)

        for conv_layer in self.conv_layers:
            conv_layer.kernel_regularizer = l2(dynamic_l2_regularization)
            conv_layer.bias_regularizer = l2(dynamic_l2_regularization)

        for d_layer in self.dropout_layers:
            d_layer.rate.assign(dynamic_dropout)

custom_learning_rate = 0.001  # Set your desired learning rate
custom_optimizer = Adam(learning_rate=custom_learning_rate)

# Define a Sequential model
model5 = Sequential()
# CNN part
conv1d_1 = Conv1D(128, 3, activation='relu', input_shape=(153, 1), kernel_regularizer=l2(0.01),
                  bias_regularizer=l2(0.01))
model5.add(conv1d_1)
model5.add(MaxPooling1D(pool_size=2))
conv1d_2 = Conv1D(64, 3, activation='relu', kernel_regularizer=l2(0.01),
                  bias_regularizer=l2(0.01))
model5.add(conv1d_2)
model5.add(MaxPooling1D(pool_size=2))
conv1d_3 = Conv1D(32, 3, activation='relu', kernel_regularizer=l2(0.01),
                  bias_regularizer=l2(0.01))
model5.add(conv1d_3)
model5.add(MaxPooling1D(pool_size=2))
# RNN part
model5.add(LSTM(512, return_sequences=True))
model5.add(Flatten())
model5.add(Dense(1024, activation='relu'))
dropout1 = DynamicDropout(0.1)
model5.add(dropout1)
# model5.add(Dense(256, activation='relu'))
# dropout2 = DynamicDropout(0.3)
# model5.add(dropout2)
model5.add(Dense(1))

# model4.add(Dense(1024, activation='relu'))
# model4.add(Dropout(0.3))
# model4.add(Dense(1))
# Compile the model
model5.compile(optimizer=custom_optimizer, loss='mean_squared_error')
model5_mse = []
model5_r2s = []
model5_mae = []
regularization_strength_values = []
dropout_rate_values=[]

# Perform k-fold cross-validation
for i, (train_index, test_index) in enumerate(kf.split(X, y)):
    X_train, X_test = X.iloc[train_index, :], X.iloc[test_index, :]
    y_train, y_test = y.iloc[train_index], y[test_index]
    # callback with the convolutional layers to apply dynamic L2 regularization
    dynamic_l2_callback = DynamicL2Regularization(conv_layers=[conv1d_1, conv1d_2, conv1d_3], dropout_layers=[dropout1])

    # dynamic_l2_callback = DynamicL2Regularization(conv_layers=[conv1d_1, conv1d_2, conv1d_3], dropout_layers=[dropout1, dropout2])

    # Fit the model with the callback
    history = model5.fit(X_train, y_train, epochs=20, callbacks=[dynamic_l2_callback], verbose=1)

    y_pred = model5.predict(X_train)
    mse = mean_squared_error(y_train, y_pred)
    r2s = r2_score(y_train, y_pred)
    mae = mean_absolute_error(y_train, y_pred)
    model5_mse.append(mse)
    model5_r2s.append(r2s)
    model5_mae.append(mae)

In [ ]:
print('r2score=',model5_r2s)
print('meansquare=',model5_mse)
print('meanabsloute=',model5_mae)
print(regularization_strength_values)
print(dropout_rate_values)

In [ ]:
pre = model5.predict(X_test)
mse = mean_squared_error(y_test, pre)
r2s = r2_score(y_test, pre)
mae = mean_absolute_error(y_test, pre)
print('r2score=',r2s)
print('meansquare=',mse)
print('meanabsloute=',mae)
model5_test={}
model5_test['r2score']=r2s
model5_test['meansquare']=mse
model5_test['meanabsloute']=mae

## RMS Prop Optimizer

In [ ]:
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Layer, Conv1D, MaxPooling1D, LSTM, Flatten, Dense, Dropout
from tensorflow.keras.regularizers import l2
from tensorflow.keras.callbacks import Callback
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.model_selection import KFold
from tensorflow.keras.optimizers import RMSprop


# kf = KFold(n_splits=5, shuffle=True,random_state=42)

import numpy as np
# from tensorflow.keras.layers import Layer
import tensorflow as tf

class DynamicDropout(Layer):
    def __init__(self, initial_rate=0.1, **kwargs):
        super(DynamicDropout, self).__init__(**kwargs)
        self.rate = tf.Variable(initial_rate, trainable=False, dtype=tf.float32)

    def call(self, inputs, training=None):
        if training:
            return tf.nn.dropout(inputs, rate=self.rate)
        return inputs

    def get_config(self):
        config = super().get_config().copy()
        config.update({"initial_rate": self.rate.numpy()})
        return config


class DynamicL2Regularization(Callback):
    def __init__(self, conv_layers, dropout_layers, initial_dropout=0.3, initial_strength=0.01):
        super(DynamicL2Regularization, self).__init__()
        self.conv_layers = conv_layers
        self.dropout_layers = dropout_layers
        # self.initial_l2 = initial_l2
        self.initial_dropout = initial_dropout
        self.initial_strength = initial_strength

    def on_epoch_end(self, epoch, logs=None):
        training_loss = logs['loss']
        d1=-training_loss
        d2=(-self.initial_strength*math.exp(-training_loss/20))/20
        gradient=d1*d2

        # Dynamic L2 and Dropout strength
        dynamic_l2_regularization = self.initial_strength - (0.001*gradient)
        dynamic_dropout=self.initial_dropout+(0.001*gradient)
        # dynamic_dropout=self.initial_dropout*math.exp(training_loss/20)

        # # Bound the values
        # dynamic_l2_regularization=max(0.001, min(0.1, dynamic_l2_regularization))
        dynamic_dropout=max(0.1, min(0.5,dynamic_dropout))


        regularization_strength_values.append(dynamic_l2_regularization)
        dropout_rate_values.append(dynamic_dropout)

        for conv_layer in self.conv_layers:
            conv_layer.kernel_regularizer = l2(dynamic_l2_regularization)
            conv_layer.bias_regularizer = l2(dynamic_l2_regularization)

        for d_layer in self.dropout_layers:
            d_layer.rate.assign(dynamic_dropout)
custom_learning_rate = 0.001  # Set your desired learning rate
custom_optimizer = RMSprop(learning_rate=custom_learning_rate)

# Define a Sequential model
model5 = Sequential()
# CNN part
conv1d_1 = Conv1D(128, 3, activation='relu', input_shape=(153, 1), kernel_regularizer=l2(0.01),
                  bias_regularizer=l2(0.01))
model5.add(conv1d_1)
model5.add(MaxPooling1D(pool_size=2))
conv1d_2 = Conv1D(64, 3, activation='relu', kernel_regularizer=l2(0.01),
                  bias_regularizer=l2(0.01))
model5.add(conv1d_2)
model5.add(MaxPooling1D(pool_size=2))
conv1d_3 = Conv1D(32, 3, activation='relu', kernel_regularizer=l2(0.01),
                  bias_regularizer=l2(0.01))
model5.add(conv1d_3)
model5.add(MaxPooling1D(pool_size=2))
# RNN part
model5.add(LSTM(512, return_sequences=True))
model5.add(Flatten())
model5.add(Dense(1024, activation='relu'))
dropout1 = DynamicDropout(0.1)
model5.add(dropout1)
# model5.add(Dense(256, activation='relu'))
# dropout2 = DynamicDropout(0.3)
# model5.add(dropout2)
model5.add(Dense(1))

# model4.add(Dense(1024, activation='relu'))
# model4.add(Dropout(0.3))
# model4.add(Dense(1))
# Compile the model
model5.compile(optimizer=custom_optimizer, loss='mean_squared_error')
model5_mse = []
model5_r2s = []
model5_mae = []
regularization_strength_values = []
dropout_rate_values=[]

# Perform k-fold cross-validation
for i, (train_index, test_index) in enumerate(kf.split(X, y)):
    X_train, X_test = X.iloc[train_index, :], X.iloc[test_index, :]
    y_train, y_test = y.iloc[train_index], y[test_index]
    # callback with the convolutional layers to apply dynamic L2 regularization

    dynamic_l2_callback = DynamicL2Regularization(conv_layers=[conv1d_1, conv1d_2, conv1d_3], dropout_layers=[dropout1])

    # Fit the model with the callback
    history = model5.fit(X_train, y_train, epochs=20, callbacks=[dynamic_l2_callback], verbose=1)
    y_pred = model5.predict(X_train)
    mse = mean_squared_error(y_train, y_pred)
    r2s = r2_score(y_train, y_pred)
    mae = mean_absolute_error(y_train, y_pred)
    model5_mse.append(mse)
    model5_r2s.append(r2s)
    model5_mae.append(mae)

In [ ]:
print('r2score=',model5_r2s)
print('meansquare=',model5_mse)
print('meanabsloute=',model5_mae)
print(regularization_strength_values)
print(dropout_rate_values)

In [ ]:
pre = model5.predict(X_test)
mse = mean_squared_error(y_test, pre)
r2s = r2_score(y_test, pre)
mae = mean_absolute_error(y_test, pre)
print('r2score=',r2s)
print('meansquare=',mse)
print('meanabsloute=',mae)
model5_test={}
model5_test['r2score']=r2s
model5_test['meansquare']=mse
model5_test['meanabsloute']=mae

## Adagrad Optimizer

In [ ]:
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Layer, Conv1D, MaxPooling1D, LSTM, Flatten, Dense, Dropout
from tensorflow.keras.regularizers import l2
from tensorflow.keras.callbacks import Callback
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.model_selection import KFold
from tensorflow.keras.optimizers import Adagrad


# kf = KFold(n_splits=5, shuffle=True,random_state=42)

import numpy as np
# from tensorflow.keras.layers import Layer
import tensorflow as tf

class DynamicDropout(Layer):
    def __init__(self, initial_rate=0.1, **kwargs):
        super(DynamicDropout, self).__init__(**kwargs)
        self.rate = tf.Variable(initial_rate, trainable=False, dtype=tf.float32)

    def call(self, inputs, training=None):
        if training:
            return tf.nn.dropout(inputs, rate=self.rate)
        return inputs

    def get_config(self):
        config = super().get_config().copy()
        config.update({"initial_rate": self.rate.numpy()})
        return config


class DynamicL2Regularization(Callback):
    def __init__(self, conv_layers, dropout_layers, initial_dropout=0.3, initial_strength=0.01):
        super(DynamicL2Regularization, self).__init__()
        self.conv_layers = conv_layers
        self.dropout_layers = dropout_layers
        # self.initial_l2 = initial_l2
        self.initial_dropout = initial_dropout
        self.initial_strength = initial_strength

    def on_epoch_end(self, epoch, logs=None):
        training_loss = logs['loss']
        d1=-training_loss
        d2=(-self.initial_strength*math.exp(-training_loss/20))/20
        gradient=d1*d2

        # Dynamic L2 and Dropout strength
        dynamic_l2_regularization = self.initial_strength - (0.001*gradient)
        dynamic_dropout=self.initial_dropout+(0.001*gradient)
        # dynamic_dropout=self.initial_dropout*math.exp(training_loss/20)

        # # Bound the values
        # dynamic_l2_regularization=max(0.001, min(0.1, dynamic_l2_regularization))
        dynamic_dropout=max(0.1, min(0.5,dynamic_dropout))


        regularization_strength_values.append(dynamic_l2_regularization)
        dropout_rate_values.append(dynamic_dropout)

        for conv_layer in self.conv_layers:
            conv_layer.kernel_regularizer = l2(dynamic_l2_regularization)
            conv_layer.bias_regularizer = l2(dynamic_l2_regularization)

        for d_layer in self.dropout_layers:
            d_layer.rate.assign(dynamic_dropout)
custom_learning_rate = 0.001  # Set your desired learning rate

custom_optimizer = Adagrad(learning_rate=custom_learning_rate)

# Define a Sequential model
model5 = Sequential()
# CNN part
conv1d_1 = Conv1D(128, 3, activation='relu', input_shape=(153, 1), kernel_regularizer=l2(0.01),
                  bias_regularizer=l2(0.01))
model5.add(conv1d_1)
model5.add(MaxPooling1D(pool_size=2))
conv1d_2 = Conv1D(64, 3, activation='relu', kernel_regularizer=l2(0.01),
                  bias_regularizer=l2(0.01))
model5.add(conv1d_2)
model5.add(MaxPooling1D(pool_size=2))
conv1d_3 = Conv1D(32, 3, activation='relu', kernel_regularizer=l2(0.01),
                  bias_regularizer=l2(0.01))
model5.add(conv1d_3)
model5.add(MaxPooling1D(pool_size=2))
# RNN part
model5.add(LSTM(512, return_sequences=True))
model5.add(Flatten())
model5.add(Dense(1024, activation='relu'))
dropout1 = DynamicDropout(0.1)
model5.add(dropout1)
# model5.add(Dense(256, activation='relu'))
# dropout2 = DynamicDropout(0.3)
# model5.add(dropout2)
model5.add(Dense(1))

# model4.add(Dense(1024, activation='relu'))
# model4.add(Dropout(0.3))
# model4.add(Dense(1))
# Compile the model
model5.compile(optimizer=custom_optimizer, loss='mean_squared_error')
model5_mse = []
model5_r2s = []
model5_mae = []
regularization_strength_values = []
dropout_rate_values=[]

# Perform k-fold cross-validation
for i, (train_index, test_index) in enumerate(kf.split(X, y)):
    X_train, X_test = X.iloc[train_index, :], X.iloc[test_index, :]
    y_train, y_test = y.iloc[train_index], y[test_index]
    # callback with the convolutional layers to apply dynamic L2 regularization
    dynamic_l2_callback = DynamicL2Regularization(conv_layers=[conv1d_1, conv1d_2, conv1d_3], dropout_layers=[dropout1])

    # Fit the model with the callback
    history = model5.fit(X_train, y_train, epochs=20, callbacks=[dynamic_l2_callback], verbose=1)
    y_pred = model5.predict(X_train)
    mse = mean_squared_error(y_train, y_pred)
    r2s = r2_score(y_train, y_pred)
    mae = mean_absolute_error(y_train, y_pred)
    model5_mse.append(mse)
    model5_r2s.append(r2s)
    model5_mae.append(mae)

In [ ]:
print('r2score=',model5_r2s)
print('meansquare=',model5_mse)
print('meanabsloute=',model5_mae)
print(regularization_strength_values)
print(dropout_rate_values)

In [ ]:
pre = model5.predict(X_test)
mse = mean_squared_error(y_test, pre)
r2s = r2_score(y_test, pre)
mae = mean_absolute_error(y_test, pre)
print('r2score=',r2s)
print('meansquare=',mse)
print('meanabsloute=',mae)
model5_test={}
model5_test['r2score']=r2s
model5_test['meansquare']=mse
model5_test['meanabsloute']=mae

## Adamax Optimizer

In [ ]:
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Layer, Conv1D, MaxPooling1D, LSTM, Flatten, Dense, Dropout
from tensorflow.keras.regularizers import l2
from tensorflow.keras.callbacks import Callback
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.model_selection import KFold
from tensorflow.keras.optimizers import Adamax


# kf = KFold(n_splits=5, shuffle=True,random_state=42)

import numpy as np
# from tensorflow.keras.layers import Layer
import tensorflow as tf

class DynamicDropout(Layer):
    def __init__(self, initial_rate=0.1, **kwargs):
        super(DynamicDropout, self).__init__(**kwargs)
        self.rate = tf.Variable(initial_rate, trainable=False, dtype=tf.float32)

    def call(self, inputs, training=None):
        if training:
            return tf.nn.dropout(inputs, rate=self.rate)
        return inputs

    def get_config(self):
        config = super().get_config().copy()
        config.update({"initial_rate": self.rate.numpy()})
        return config


class DynamicL2Regularization(Callback):
    def __init__(self, conv_layers, dropout_layers, initial_dropout=0.3, initial_strength=0.01):
        super(DynamicL2Regularization, self).__init__()
        self.conv_layers = conv_layers
        self.dropout_layers = dropout_layers
        # self.initial_l2 = initial_l2
        self.initial_dropout = initial_dropout
        self.initial_strength = initial_strength

    def on_epoch_end(self, epoch, logs=None):
        training_loss = logs['loss']
        d1=-training_loss
        d2=(-self.initial_strength*math.exp(-training_loss/20))/20
        gradient=d1*d2

        # Dynamic L2 and Dropout strength
        dynamic_l2_regularization = self.initial_strength - (0.001*gradient)
        dynamic_dropout=self.initial_dropout+(0.001*gradient)
        # dynamic_dropout=self.initial_dropout*math.exp(training_loss/20)

        # # Bound the values
        # dynamic_l2_regularization=max(0.001, min(0.1, dynamic_l2_regularization))
        dynamic_dropout=max(0.1, min(0.5,dynamic_dropout))


        regularization_strength_values.append(dynamic_l2_regularization)
        dropout_rate_values.append(dynamic_dropout)

        for conv_layer in self.conv_layers:
            conv_layer.kernel_regularizer = l2(dynamic_l2_regularization)
            conv_layer.bias_regularizer = l2(dynamic_l2_regularization)

        for d_layer in self.dropout_layers:
            d_layer.rate.assign(dynamic_dropout)
custom_learning_rate = 0.001  # Set your desired learning rate

custom_optimizer = Adamax(learning_rate=custom_learning_rate)

# Define a Sequential model
model5 = Sequential()
# CNN part
conv1d_1 = Conv1D(128, 3, activation='relu', input_shape=(153, 1), kernel_regularizer=l2(0.01),
                  bias_regularizer=l2(0.01))
model5.add(conv1d_1)
model5.add(MaxPooling1D(pool_size=2))
conv1d_2 = Conv1D(64, 3, activation='relu', kernel_regularizer=l2(0.01),
                  bias_regularizer=l2(0.01))
model5.add(conv1d_2)
model5.add(MaxPooling1D(pool_size=2))
conv1d_3 = Conv1D(32, 3, activation='relu', kernel_regularizer=l2(0.01),
                  bias_regularizer=l2(0.01))
model5.add(conv1d_3)
model5.add(MaxPooling1D(pool_size=2))
# RNN part
model5.add(LSTM(512, return_sequences=True))
model5.add(Flatten())
model5.add(Dense(1024, activation='relu'))
dropout1 = DynamicDropout(0.1)
model5.add(dropout1)
# model5.add(Dense(256, activation='relu'))
# dropout2 = DynamicDropout(0.3)
# model5.add(dropout2)
model5.add(Dense(1))

# model4.add(Dense(1024, activation='relu'))
# model4.add(Dropout(0.3))
# model4.add(Dense(1))
# Compile the model
model5.compile(optimizer=custom_optimizer, loss='mean_squared_error')
model5_mse = []
model5_r2s = []
model5_mae = []
regularization_strength_values = []
dropout_rate_values=[]

# Perform k-fold cross-validation
for i, (train_index, test_index) in enumerate(kf.split(X, y)):
    X_train, X_test = X.iloc[train_index, :], X.iloc[test_index, :]
    y_train, y_test = y.iloc[train_index], y[test_index]
    # callback with the convolutional layers to apply dynamic L2 regularization
    dynamic_l2_callback = DynamicL2Regularization(conv_layers=[conv1d_1, conv1d_2, conv1d_3], dropout_layers=[dropout1])
    # Fit the model with the callback
    history = model5.fit(X_train, y_train, epochs=20, callbacks=[dynamic_l2_callback], verbose=1)
    y_pred = model5.predict(X_train)
    mse = mean_squared_error(y_train, y_pred)
    r2s = r2_score(y_train, y_pred)
    mae = mean_absolute_error(y_train, y_pred)
    model5_mse.append(mse)
    model5_r2s.append(r2s)
    model5_mae.append(mae)

In [ ]:
print('r2score=',model5_r2s)
print('meansquare=',model5_mse)
print('meanabsloute=',model5_mae)
print(regularization_strength_values)
print(dropout_rate_values)

In [ ]:
pre = model5.predict(X_test)
mse = mean_squared_error(y_test, pre)
r2s = r2_score(y_test, pre)
mae = mean_absolute_error(y_test, pre)
print('r2score=',r2s)
print('meansquare=',mse)
print('meanabsloute=',mae)
model5_test={}
model5_test['r2score']=r2s
model5_test['meansquare']=mse
model5_test['meanabsloute']=mae

## SDG Optimizer

In [ ]:
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Layer, Conv1D, MaxPooling1D, LSTM, Flatten, Dense, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.regularizers import l2
from tensorflow.keras.callbacks import Callback
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.model_selection import KFold
from tensorflow.keras.optimizers import SGD


# kf = KFold(n_splits=5, shuffle=True,random_state=42)

import numpy as np
# from tensorflow.keras.layers import Layer
import tensorflow as tf

class DynamicDropout(Layer):
    def __init__(self, initial_rate=0.1, **kwargs):
        super(DynamicDropout, self).__init__(**kwargs)
        self.rate = tf.Variable(initial_rate, trainable=False, dtype=tf.float32)

    def call(self, inputs, training=None):
        if training:
            return tf.nn.dropout(inputs, rate=self.rate)
        return inputs

    def get_config(self):
        config = super().get_config().copy()
        config.update({"initial_rate": self.rate.numpy()})
        return config


class DynamicL2Regularization(Callback):
    def __init__(self, conv_layers, dropout_layers, initial_dropout=0.3, initial_strength=0.01):
        super(DynamicL2Regularization, self).__init__()
        self.conv_layers = conv_layers
        self.dropout_layers = dropout_layers
        # self.initial_l2 = initial_l2
        self.initial_dropout = initial_dropout
        self.initial_strength = initial_strength

    def on_epoch_end(self, epoch, logs=None):
        training_loss = logs['loss']
        d1=-training_loss
        d2=(-self.initial_strength*math.exp(-training_loss/20))/20
        gradient=d1*d2

        # Dynamic L2 and Dropout strength
        dynamic_l2_regularization = self.initial_strength - (0.001*gradient)
        dynamic_dropout=self.initial_dropout+(0.001*gradient)
        # dynamic_dropout=self.initial_dropout*math.exp(training_loss/20)

        # # Bound the values
        # dynamic_l2_regularization=max(0.001, min(0.1, dynamic_l2_regularization))
        dynamic_dropout=max(0.1, min(0.5,dynamic_dropout))


        regularization_strength_values.append(dynamic_l2_regularization)
        dropout_rate_values.append(dynamic_dropout)

        for conv_layer in self.conv_layers:
            conv_layer.kernel_regularizer = l2(dynamic_l2_regularization)
            conv_layer.bias_regularizer = l2(dynamic_l2_regularization)

        for d_layer in self.dropout_layers:
            d_layer.rate.assign(dynamic_dropout)
custom_learning_rate = 0.001  # Set your desired learning rate
custom_optimizer = SGD(learning_rate=custom_learning_rate)

# Define a Sequential model
model5 = Sequential()
# CNN part
conv1d_1 = Conv1D(128, 3, activation='relu', input_shape=(153, 1), kernel_regularizer=l2(0.01),
                  bias_regularizer=l2(0.01))
model5.add(conv1d_1)
model5.add(MaxPooling1D(pool_size=2))
conv1d_2 = Conv1D(64, 3, activation='relu', kernel_regularizer=l2(0.01),
                  bias_regularizer=l2(0.01))
model5.add(conv1d_2)
model5.add(MaxPooling1D(pool_size=2))
conv1d_3 = Conv1D(32, 3, activation='relu', kernel_regularizer=l2(0.01),
                  bias_regularizer=l2(0.01))
model5.add(conv1d_3)
model5.add(MaxPooling1D(pool_size=2))
# RNN part
model5.add(LSTM(512, return_sequences=True))
model5.add(Flatten())
model5.add(Dense(1024, activation='relu'))
dropout1 = DynamicDropout(0.1)
model5.add(dropout1)
# model5.add(Dense(256, activation='relu'))
# dropout2 = DynamicDropout(0.3)
# model5.add(dropout2)
model5.add(Dense(1))

# model4.add(Dense(1024, activation='relu'))
# model4.add(Dropout(0.3))
# model4.add(Dense(1))
# Compile the model
model5.compile(optimizer=custom_optimizer, loss='mean_squared_error')
model5_mse = []
model5_r2s = []
model5_mae = []
regularization_strength_values = []
dropout_rate_values=[]

# Perform k-fold cross-validation
for i, (train_index, test_index) in enumerate(kf.split(X, y)):
    X_train, X_test = X.iloc[train_index, :], X.iloc[test_index, :]
    y_train, y_test = y.iloc[train_index], y[test_index]
    # callback with the convolutional layers to apply dynamic L2 regularization
    dynamic_l2_callback = DynamicL2Regularization(conv_layers=[conv1d_1, conv1d_2, conv1d_3], dropout_layers=[dropout1])
    # Fit the model with the callback
    history = model5.fit(X_train, y_train, epochs=20, callbacks=[dynamic_l2_callback], verbose=1)
    y_pred = model5.predict(X_train)
    mse = mean_squared_error(y_train, y_pred)
    r2s = r2_score(y_train, y_pred)
    mae = mean_absolute_error(y_train, y_pred)
    model5_mse.append(mse)
    model5_r2s.append(r2s)
    model5_mae.append(mae)

In [ ]:
print('r2score=',model5_r2s)
print('meansquare=',model5_mse)
print('meanabsloute=',model5_mae)
print(regularization_strength_values)
print(dropout_rate_values)

In [ ]:
pre = model5.predict(X_test)
mse = mean_squared_error(y_test, pre)
r2s = r2_score(y_test, pre)
mae = mean_absolute_error(y_test, pre)
print('r2score=',r2s)
print('meansquare=',mse)
print('meanabsloute=',mae)
model5_test={}
model5_test['r2score']=r2s
model5_test['meansquare']=mse
model5_test['meanabsloute']=mae